# Census: Clasificación Multiclase Multivariable con PyTorch

Este notebook entrena una red neuronal en PyTorch para **clasificación multiclase** usando todas las variables del dataset `adult.csv` (excepto la variable objetivo).

Objetivo elegido: `occupation` (múltiples clases).

Métricas calculadas: `accuracy`, `precision`, `recall`, `f1` (macro y weighted), matriz de confusión y `roc_auc_ovr_macro`.

In [ ]:
# Si falta PyTorch en tu entorno, descomenta la siguiente línea:
# %pip install torch

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# 1) Cargar, limpiar y definir target multiclase
df = pd.read_csv('adult.csv').copy()
df.columns = [c.strip() for c in df.columns]

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()

# '?' representa valor faltante
df = df.replace('?', np.nan)

target_col = 'occupation'
if target_col not in df.columns:
    raise ValueError('No se encontró la columna target occupation en adult.csv')

# Para clasificación multiclase removemos filas sin etiqueta target
df = df.dropna(subset=[target_col]).copy()

X = df.drop(columns=[target_col])
y_raw = df[target_col].astype(str)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
class_names = label_encoder.classes_
num_classes = len(class_names)

print('Shape total X:', X.shape)
print('Número de clases:', num_classes)
print('Clases:', list(class_names))

In [ ]:
# 2) Split + preprocesamiento multivariable (num + cat)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

num_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols),
    ]
)

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

if hasattr(X_train_t, 'toarray'):
    X_train_t = X_train_t.toarray()
if hasattr(X_test_t, 'toarray'):
    X_test_t = X_test_t.toarray()

X_train_t = X_train_t.astype(np.float32)
X_test_t = X_test_t.astype(np.float32)

y_train_np = y_train.astype(np.int64)
y_test_np = y_test.astype(np.int64)

print('X_train transformado:', X_train_t.shape)
print('X_test transformado:', X_test_t.shape)

In [ ]:
# 3) Definir red multiclase y DataLoaders
train_ds = TensorDataset(
    torch.from_numpy(X_train_t),
    torch.from_numpy(y_train_np)
 )
test_ds = TensorDataset(
    torch.from_numpy(X_test_t),
    torch.from_numpy(y_test_np)
 )

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)

input_dim = X_train_t.shape[1]

class CensusMLP(nn.Module):
    def __init__(self, in_features, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        return self.net(x)

model = CensusMLP(input_dim, num_classes).to(device)

# Pesos por clase para manejar desbalanceo
class_counts = np.bincount(y_train_np, minlength=num_classes)
class_weights = class_counts.sum() / np.maximum(class_counts, 1)
class_weights = class_weights / class_weights.mean()
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights_t)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print('input_dim:', input_dim)
print('num_classes:', num_classes)

In [ ]:
# 4) Entrenamiento completo
EPOCHS = 15
history = {'train_loss': [], 'test_loss': [], 'test_f1_macro': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    test_losses = []
    all_preds = []
    all_true = []

    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            preds = torch.argmax(logits, dim=1)
            test_losses.append(loss.item())
            all_preds.extend(preds.detach().cpu().numpy().tolist())
            all_true.extend(yb.detach().cpu().numpy().tolist())

    train_loss = float(np.mean(train_losses))
    test_loss = float(np.mean(test_losses))
    f1_macro = f1_score(all_true, all_preds, average='macro')

    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    history['test_f1_macro'].append(f1_macro)

    print(f'Epoch {epoch:02d}/{EPOCHS} | train_loss={train_loss:.4f} | test_loss={test_loss:.4f} | test_f1_macro={f1_macro:.4f}')

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history['train_loss'], label='train_loss')
plt.plot(history['test_loss'], label='test_loss')
plt.title('Curvas de pérdida')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history['test_f1_macro'], label='test_f1_macro')
plt.title('F1 macro en test')
plt.xlabel('Epoch')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 5) Preparar predicciones para evaluar métricas (multiclase)
model.eval()
with torch.no_grad():
    X_test_tensor = torch.from_numpy(X_test_t).to(device)
    logits = model(X_test_tensor)
    y_proba = torch.softmax(logits, dim=1).cpu().numpy()

y_pred = np.argmax(y_proba, axis=1)
y_true = y_test_np

print('Predicciones listas para métricas.')
print('Shape y_true:', y_true.shape, '| Shape y_pred:', y_pred.shape, '| Shape y_proba:', y_proba.shape)

### Accuracy

In [ ]:
def accuracy_multiclass(y_true, y_pred):
    return accuracy_score(y_true, y_pred)

acc = accuracy_multiclass(y_true, y_pred)
print(f'Accuracy: {acc:.4f}')

### Precision y Recall

In [ ]:
def precision_multiclass(y_true, y_pred, average='macro'):
    return precision_score(y_true, y_pred, average=average, zero_division=0)

def recall_multiclass(y_true, y_pred, average='macro'):
    return recall_score(y_true, y_pred, average=average, zero_division=0)

p_macro = precision_multiclass(y_true, y_pred, average='macro')
r_macro = recall_multiclass(y_true, y_pred, average='macro')
p_weighted = precision_multiclass(y_true, y_pred, average='weighted')
r_weighted = recall_multiclass(y_true, y_pred, average='weighted')

print(f'Precision macro: {p_macro:.4f} | Recall macro: {r_macro:.4f}')
print(f'Precision weighted: {p_weighted:.4f} | Recall weighted: {r_weighted:.4f}')

### Matriz de Confusión

In [ ]:
def confusion_matrix_multiclass(y_true, y_pred):
    return confusion_matrix(y_true, y_pred)

cm = confusion_matrix_multiclass(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(12, 10))
disp.plot(cmap='Blues', xticks_rotation=90, ax=ax, colorbar=False)
plt.title('Matriz de Confusión - PyTorch MLP (Multiclase)')
plt.tight_layout()
plt.show()

### ROC-AUC (One-vs-Rest)

In [ ]:
def roc_auc_ovr_macro(y_true, y_proba):
    try:
        return roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
    except ValueError:
        return np.nan

auc_ovr_macro = roc_auc_ovr_macro(y_true, y_proba)
print(f'ROC-AUC OVR macro: {auc_ovr_macro:.4f}')

print('\nReporte de clasificación:\n')
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

### F1-Score

In [ ]:
def f1_multiclass(y_true, y_pred, average='macro'):
    return f1_score(y_true, y_pred, average=average, zero_division=0)

f1_macro = f1_multiclass(y_true, y_pred, average='macro')
f1_weighted = f1_multiclass(y_true, y_pred, average='weighted')

print(f'F1 macro: {f1_macro:.4f}')
print(f'F1 weighted: {f1_weighted:.4f}')

### Top clases por F1

In [ ]:
# 6) Top clases con mayor F1 (resumen rápido)
from sklearn.metrics import f1_score

f1_por_clase = f1_score(y_true, y_pred, average=None, zero_division=0)
df_f1_clase = pd.DataFrame({
    'clase': class_names,
    'f1': f1_por_clase
}).sort_values('f1', ascending=False)

print('Top 10 clases por F1:')
print(df_f1_clase.head(10))

print('\nBottom 10 clases por F1:')
print(df_f1_clase.tail(10))